# ZizAIの初回セットアップ（Windows / VS Code）

**最初に選ぶカーネルは、このプロジェクトの `.venv\Scripts\python.exe` です。** 新規の場合は、Notebookのコードセルを実行する前に作成します。

1. VS Codeで `pyproject.toml` があるフォルダーを開きます。Python拡張とJupyter拡張が必要です。
2. VS Codeの **ターミナル → 新しいターミナル** でPowerShellを開き、そのフォルダーで次の2行を実行します。`py -3.14` はインストール済みのPython 3.14を使う指定です。

```powershell
py -3.14 -m venv .venv
& ".\.venv\Scripts\python.exe" -m pip install ipykernel
```

すでにプロジェクトの `.venv` がある場合は、1行目は省略します。`py -3.14` が見つからなければ `py -0p` で確認し、手元のPython 3.12以降を指定してください。

3. このNotebookの右上で **カーネルの選択 → 他のカーネルを選択 → Python環境** を開き、`.venv\Scripts\python.exe` を選びます。[VS Codeのカーネル選択手順](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)
4. 最初のコードセル **0** の左にある **▶** を押します。**0 → ① → ② → ③** と1セルずつ実行し、③の後で同じカーネルを再起動します。④は再起動後に実行します。

macOS・Linuxでは、作成に `python3 -m venv .venv`（Python 3.12以降）、`ipykernel` の導入に `.venv/bin/python -m pip install ipykernel` を使い、カーネルは `.venv/bin/python` を選びます。

## 0. セットアップの設定を確認する

初回はこのセルをそのまま実行してください。選択した `.venv` にフォームとCSVを導入します。

追加するコネクタは `connector_extras` で指定します。指定できる値は `csv`、`excel`、`duckdb`、`bigquery`、`dataintegration`、`python`、`shell`、`vector`、`windows` です。別のコネクタ共通フォルダーを使う場合は `shared_connectors` を指定します。

すでにuvを導入済みで、ZizAIとライブラリのインストールに使う場合は `installer = "uv"` にします。既存の別環境を利用する場合は、その環境に `ipykernel` があることを確認してNotebook右上で選び、`use_project_venv = False` にします。②と③の対象は手順0に表示されます。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tomllib

project_root = None              # 自動判定できないときは例: "C:/tools/ZizAI"
shared_connectors = None         # 省略時は ZizAI/connectors
use_project_venv = True
installer = "pip"                # または "uv"
connector_extras = ["csv"]       # 例: ["csv", "excel", "duckdb"]

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以降のカーネルを選択してください。")

start = Path(project_root or Path.cwd()).expanduser().resolve()
roots = (start,) if project_root else (start, *start.parents)
root = next((p for p in roots if (p / "pyproject.toml").is_file()
             and (p / "src/forms.py").is_file() and (p / "connectors").is_dir()), None)
if root is None:
    raise ValueError("ZizAIのルートを開くか、project_rootにルートのパスを指定してください。")

catalog = Path(shared_connectors or root / "connectors").expanduser().resolve()
if not catalog.is_dir() or not any(catalog.glob("*/*_definition.yaml")):
    raise ValueError("コネクタYAMLがある共通フォルダーを指定してください。")

project = tomllib.loads((root / "pyproject.toml").read_text(encoding="utf-8"))["project"]
extras = list(dict.fromkeys(["forms", *connector_extras]))
unknown = set(extras) - project["optional-dependencies"].keys()
if unknown:
    raise ValueError(f"不明な追加ライブラリ指定: {', '.join(sorted(unknown))}")
if installer not in {"pip", "uv"}:
    raise ValueError('installerは "pip" または "uv" にしてください。')

environment = root / ".venv" if use_project_venv else Path(sys.prefix)
python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if use_project_venv:
    if environment.is_symlink() or environment.is_junction():
        raise ValueError(".venvがリンクです。既存環境として利用する場合は、対象カーネルを選びuse_project_venv = Falseにしてください。")
    if not (environment / "pyvenv.cfg").is_file() or not python.is_file():
        raise RuntimeError("先に冒頭のターミナル手順で.venvを作ってください。")
    if Path(sys.prefix).resolve() != environment.resolve():
        raise RuntimeError(f"Notebook右上のカーネル選択で、このPythonを選んでください: {python}")
else:
    python = Path(sys.executable)

def run_target(*args, **kwargs):
    return subprocess.run([str(python), "-I", *args], check=True, **kwargs)

print(f"ZizAI: {root}")
print(f"導入先のPython: {python}")
print(f"共通コネクタ: {catalog}")
print(f"インストーラー: {installer} / 追加ライブラリ: {', '.join(extras)}")

## ① 選択したPythonを確認する

このセルでPythonのバージョンとインストール先を確認します。`準備済み` と表示されたら②へ進んでください。

In [ ]:
if installer == "uv" and not shutil.which("uv"):
    raise RuntimeError('uvが見つかりません。先に導入するか、0で installer = "pip" にしてください。')

run_target("-c", "from pathlib import Path; import sys; "
           "assert sys.version_info >= (3, 12), 'Python 3.12以上が必要です。'; "
           "assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve(), sys.prefix",
           str(environment))
print(f"準備済み: {python}")

## ② ZizAIとライブラリを導入する

①で確認したPythonへZizAI本体と選択した追加ライブラリを導入し、ipykernelも確認します。通常はパッケージ取得に通信が必要です。Vectorのモデル取得やBigQueryへの接続は、ここでは行いません。

In [ ]:
target = f"{root}[{','.join(extras)}]"
if installer == "uv":
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uvが見つかりません。セル0の設定を確認してください。")
    subprocess.run([uv, "pip", "install", "--python", str(python), "ipykernel", target], check=True)
else:
    run_target("-m", "pip", "--isolated", "install", "ipykernel", target)

run_target("-c", "import ipykernel, importlib.metadata as m; "
           "print('ZizAI:', m.version('zizai-runtime'))")

## ③ コネクタ共通フォルダーを登録する

選んだPython環境に共通フォルダーの場所を登録します。既存の登録先を切り替える場合は、セル0の表示を確認してから実行してください。登録が終わったら次の案内に従って同じカーネルを再起動します。

In [ ]:
registration_env = os.environ.copy()
for name in ("ZIZAI_CONFIG_FILE", "ZIZAI_CONNECTORS_DIR"):
    registration_env.pop(name, None)

run_target("-c", "import sys; from zizai.connectors import configure_connectors; "
           "print('登録先:', configure_connectors(sys.argv[1]))",
           catalog.as_posix(), env=registration_env)
print(f"このPythonカーネルを再起動してください: {python}")

## ③の後で、同じカーネルを再起動する

1. Notebook上部の **再起動（Restart）** を押します。右上は、最初に選んだ `.venv` のままです。Pythonは再起動時に③の登録を読み込みます。
2. **0のコードセルをもう一度実行**します。①〜③は実行済みです。
3. ④のコードセルを上から順に実行します。最後の `%form` でフォームが表示されます。

## ④ マジックを読み込み、フォームを表示する

ここで使うマジックは、通常のNotebookでもカーネルを再起動するたびに読み込みます。フォームで生成したコードは、必要なときに別のセルへコピーして使います。

In [ ]:
if "environment" not in globals():
    raise RuntimeError("カーネル再起動後にセル0を実行してください。")
if Path(sys.prefix).resolve() != environment.resolve():
    raise RuntimeError(f"③で表示されたPythonをカーネルに選び直してください: {python}")

from zizai.core.connector_config import get_connector_directory
from zizai.form_definition import load_definition

if get_connector_directory() != catalog:
    raise RuntimeError("③で登録したコネクタの場所と異なります。登録先とZIZAI_CONFIG_FILE・ZIZAI_CONNECTORS_DIRを確認してください。")

definitions = load_definition()
print(f"使用中のPython: {sys.executable}")
print("コネクタ登録を確認しました。下の2セルを実行してください。")

In [ ]:
%load_ext zizai.forms

In [ ]:
%form

## ⑤ VS Codeでフォームを呼び出すスニペットを登録する（任意）

別のNotebookでもフォームを手軽に開きたい場合は、下のセルを一度実行します。VS Codeの標準ユーザープロファイルに `@zizai_form` を登録します。

Pythonセルで `@zizai_form` と入力し、候補の「ZizAIフォームを開く」を選ぶと、`%load_ext zizai.forms` と `%form` が挿入されます。候補が出ないときは **Ctrl+Space** を押してください。挿入したセルを実行するとフォームが開きます。

In [ ]:
from pathlib import Path
import os
import sys

if "root" not in globals():
    raise RuntimeError("先にセル0を実行してください。")

if sys.platform == "win32":
    settings_home = Path(os.environ["APPDATA"])
elif sys.platform == "darwin":
    settings_home = Path.home() / "Library" / "Application Support"
else:
    settings_home = Path(os.environ.get("XDG_CONFIG_HOME", Path.home() / ".config"))

source = root / "examples" / "zizai-notebook.code-snippets"
snippet = source.read_text(encoding="utf-8-sig")
user_folders = [settings_home / edition / "User" for edition in ("Code", "Code - Insiders")]
user_folders = [folder for folder in user_folders if folder.is_dir()]
if not user_folders:
    raise FileNotFoundError(f"VS Codeのユーザー設定フォルダーを確認してください: {settings_home}")

for user_folder in user_folders:
    folder = user_folder / "snippets"
    folder.mkdir(exist_ok=True)
    destination = folder / source.name
    try:
        with destination.open("x", encoding="utf-8", newline="\n") as file:
            file.write(snippet)
    except FileExistsError:
        if destination.read_text(encoding="utf-8-sig") != snippet:
            print(f"別の内容が登録済みです。既存のファイルを保持しました: {destination}")
            continue
    print(f"@zizai_formを利用できます: {destination}")

## Python・Kiroから直接使う場合

同じPython環境で、例えば `import csv_connector` として使えます。マジックの読み込みは、Notebookでフォームを表示する場合に必要です。[導入手順](../docs/usage.md)・[Kiro向けガイド](../.kiro/steering/zizai.md)。